# Getting arxiv documents

!uv pip install arxiv mistralai

In [ ]:
import arxiv

query = "transformer neural network"

search = arxiv.Search(
    query=query,
    max_results=5,
    sort_by=arxiv.SortCriterion.Relevance
)

for result in search.results():
    print(result.title)
    print(result.pdf_url)
    print()

In [ ]:
dir(arxiv.SortCriterion)

In [ ]:
import arxiv

def search_arxiv(topic, k=3):
    search = arxiv.Search(
        query=topic,
        max_results=k,
        sort_by=arxiv.SortCriterion.Relevance
    )

    for result in search.results():
        print("TITLE:", result.title)
        print("AUTHORS:", [a.name for a in result.authors])
        print("SUMMARY:", result.summary)
        print("PUBLISHED:", result.published)
        print("UPDATED:", result.updated)
        print("PDF URL:", result.pdf_url)
        print("ARXIV URL:", result.entry_id)
        print("PRIMARY CATEGORY:", result.primary_category)
        print("CATEGORIES:", result.categories)
        print("DOI:", result.doi)
        print("COMMENT:", result.comment)
        print("JOURNAL REF:", result.journal_ref)
        print("LINKS:", result.links)
        print("-" * 80)

In [ ]:
search_arxiv("transformer neural networks", 2)

In [ ]:
from pydantic import BaseModel
from datetime import datetime
import arxiv


class ArxivDocument(BaseModel):
    title: str
    authors: list[str]
    summary: str
    published: datetime
    updated: datetime
    pdf_url: str
    arxiv_url: str
    primary_category: str
    categories: list[str]
    doi: str | None
    comment: str | None
    journal_ref: str | None

    class Config:
        arbitrary_types_allowed = True


class ArxivDocuments(BaseModel):
    documents: list[ArxivDocument]


def search_arxiv_documents(topic: str, k: int = 5) -> ArxivDocuments:
    """Search arXiv for papers matching a topic and return structured results.

    Args:
        topic: Search query string (e.g. "transformer neural networks").
        k: Maximum number of results to return.

    Returns:
        ArxivDocuments containing up to k matching papers.
    """
    client = arxiv.Client()
    search = arxiv.Search(
        query=topic,
        max_results=k,
        sort_by=arxiv.SortCriterion.Relevance,
    )

    documents = []
    for result in client.results(search):
        doc = ArxivDocument(
            title=result.title,
            authors=[a.name for a in result.authors],
            summary=result.summary,
            published=result.published,
            updated=result.updated,
            pdf_url=result.pdf_url,
            arxiv_url=result.entry_id,
            primary_category=result.primary_category,
            categories=result.categories,
            doi=result.doi,
            comment=result.comment,
            journal_ref=result.journal_ref,
        )
        documents.append(doc)

    return ArxivDocuments(documents=documents)

In [ ]:
results = search_arxiv_documents("transformer neural networks", k=2)

for doc in results.documents:
    print(f"Title: {doc.title}")
    print(f"Authors: {doc.authors}")
    print(f"Published: {doc.published}")
    print(f"PDF: {doc.pdf_url}")
    print("-" * 80)

## Downloading Arxiv pdf Document

In [ ]:
!mkdir arxiv-papers

In [ ]:
import arxiv

search = arxiv.Search(query="transformers", max_results=1)

for result in search.results():
    result.download_pdf(dirpath="./arxiv-papers")

In [ ]:
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(override=True)

In [ ]:
import os
from mistralai import Mistral
from pathlib import Path
import base64


def mistral_pdf_to_markdown(pdf_input: str, api_key: str):
    """
    Extract markdown text from a PDF using Mistral OCR.

    pdf_input can be:
        - a PDF URL
        - a local PDF file path

    Returns:
        dict with page markdown + full document markdown
    """

    client = Mistral(api_key=api_key)

    # Detect if input is URL or local file
    if pdf_input.startswith("http://") or pdf_input.startswith("https://"):

        document = {
            "type": "document_url",
            "document_url": pdf_input
        }

    else:
        pdf_path = Path(pdf_input)

        with open(pdf_path, "rb") as f:
            pdf_base64 = base64.b64encode(f.read()).decode("utf-8")

        document = {
            "type": "document_base64",
            "document_base64": pdf_base64
        }

    response = client.ocr.process(
        model="mistral-ocr-latest",
        document=document
    )

    pages_markdown = []
    for page in response.pages:
        pages_markdown.append(page.markdown)

    return {
        "pages": pages_markdown,
        "document_markdown": "\n\n".join(pages_markdown)
    }

In [ ]:
url = "https://arxiv.org/pdf/1609.04846v1"

In [ ]:
mistral_api_key = os.getenv("MISTRAL_API_KEY")

paper = mistral_pdf_to_markdown(
    url,
    mistral_api_key
)

display(Markdown(paper["document_markdown"][:2000]))

In [ ]:
len(paper["pages"])

In [ ]:
display(Markdown(paper["pages"][0]))

In [ ]:
display(Markdown(paper["pages"][1]))


In [ ]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from IPython.display import Markdown, display

from ingest import fetch_arxiv_documents, enrich_documents, create_chunks, create_embeddings

### Get Arxiv Documenys and create embeddings

In [ ]:
topics = ["RAG Systems", "transformer neural networks", "machine learning", "artificial intelligence", "AI Engineering"]
k = 5
all_documents = []
all_chunks = []

for topic in tqdm(topics[:2]):
    documents = fetch_arxiv_documents(topic, k)
    documents = enrich_documents(documents)
    all_documents.extend(documents)
    all_chunks.extend(create_chunks(documents))

In [ ]:
len(all_documents), len(all_chunks)

In [ ]:
all_documents[:5]

In [ ]:
all_chunks[:2]

In [ ]:
all_chunks[-1].metadata

In [ ]:
def clean_metadata(chunks: list) -> list:
    """Replace None values in chunk metadata with 'N/A'.
    ChromaDB requires all metadata values to be non-None (str, int, float, or bool).
    """
    for chunk in chunks:
        chunk.metadata = {
            k: v if v is not None else "N/A"
            for k, v in chunk.metadata.items()
        }
    return chunks

all_cleaned_chunks = clean_metadata(all_chunks)
print(f"Cleaned {len(all_cleaned_chunks)} chunks")

In [ ]:
create_embeddings(all_cleaned_chunks)

### Visualize Embeddings

In [ ]:
DB_NAME =  "arxiv_db"
COLLECTION_NAME = "arxiv_papers"
EMBEDDING_MODEL = "text-embedding-3-large"


In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(COLLECTION_NAME)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])

vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
titles = [m['title'] for m in metadatas]
categories = [m['primary_category'] for m in metadatas]

unique_categories = sorted(set(categories))
color_palette = ['blue', 'green', 'red', 'orange', 'purple', 'brown', 'pink', 'cyan', 'magenta', 'olive']
category_colors = {cat: color_palette[i % len(color_palette)] for i, cat in enumerate(unique_categories)}
colors = [category_colors[cat] for cat in categories]

print(f"Loaded {len(vectors)} vectors across {len(unique_categories)} categories: {unique_categories}")

In [ ]:
tsne_2d = TSNE(n_components=2, random_state=42)
reduced_2d = tsne_2d.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_2d[:, 0],
    y=reduced_2d[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Category: {cat}<br>Title: {title}<br>Text: {doc[:100]}..."
          for cat, title, doc in zip(categories, titles, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D arXiv Embeddings Visualization (t-SNE)',
    xaxis_title='x',
    yaxis_title='y',
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
tsne_3d = TSNE(n_components=3, random_state=42)
reduced_3d = tsne_3d.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter3d(
    x=reduced_3d[:, 0],
    y=reduced_3d[:, 1],
    z=reduced_3d[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Category: {cat}<br>Title: {title}<br>Text: {doc[:100]}..."
          for cat, title, doc in zip(categories, titles, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D arXiv Embeddings Visualization (t-SNE)',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()